In [ ]:
# import needed packages

#update reading in packages when rerunning this cell
%load_ext autoreload
%autoreload 2

import numpy as np
import xarray as xr 
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
from matplotlib import colormaps
from mpl_toolkits.axes_grid1 import Divider, Size

import cartopy.crs as ccrs #for plotting on map
import cartopy as cart 
import scipy.special as sc
import scipy.integrate as integrate
from datetime import datetime, timedelta
from scipy.interpolate import CubicSpline # maybe not needed
import sys
sys.path.append("/nethome/4291387/Maxey_Riley_advection/Maxey_Riley_advection/simulations")
sys.path.append("/nethome/4291387/Maxey_Riley_advection/Maxey_Riley_advection/src")
from particle_characteristics_functions import stokes_relaxation_time, Re_particle, factor_drag_white1991, diffusion_time, slip_force, dynamic_viscosity_Sharqawy
from history_term_functions import Basset_kernel, Mei1992_kernel, history_timescale, Daitche, Hinsberg, f_Mei1992, History_Force_Hinsberg_Mei_kernel, History_Force_Hinsberg_Basset_kernel
from analysis_functions_xr import derivative_backward, calc_tidal_av
from analysis_functions import make_PDF, make_lognormal_PDF
plt.style.use('../python_style_Meike.mplstyle')

Magma = colormaps['magma']
Magmalist = Magma(np.linspace(0.2, 0.9, 5))
c1_mei =2
c2_mei = 0.1015
c1_kim = 2.5
c2_kim = 0.126
c1_dorgan = 2.5
c2_dorgan = 0.2

In [ ]:
# needed constants
av_temp_NWES = 11.983276
av_salinity_NWES = 34.70449
rho_water = 1027 # kg/m3 
dynamic_viscosity_water = dynamic_viscosity_Sharqawy(av_temp_NWES,av_salinity_NWES/1000)# 1.e-3# 1.41 * 10**(-3) # kg/(ms) https://www.engineeringtoolbox.com/sea-water-properties-d_840.html (at 10 deg)
kinematic_viscosity_water = dynamic_viscosity_water / rho_water
kinematic_viscosity_eddy = 1e-3
diameter = 0.25 # m



In [ ]:
# analytical test functions

def dUslip_dt_cos(t):
    return np.cos(t,dtype = np.float128)
    
def Basset_analytical_cos(t):
    
    S, C = sc.fresnel(np.sqrt(2 * t / np.pi))
    return np.sqrt(2 * np.pi ) * (np.cos(t) * C + np.sin(t) * S )

    
def dUslip_dt_tt(t):
    return t*t
    
def Basset_analytical_tt(t):
    return 2/15 * np.sqrt(t) * 8 * t *t 

def Basset_kernel(s,t):
    return 1/np.sqrt(t-s)

def Mei_kernel(s,t,Rep,nu,d, c1, c2):
    fh = (0.75 + c2 * Rep) 
    A = nu * Rep*Rep / (fh*fh * d * d)
    term2 = ( np.sqrt(np.pi)/2 *  (A * (t - s))**(3/2) ) ** (1.0 / c1)
    return 1/np.sqrt(t-s) * ((1 + term2) ** (-c1))

def prefactor(B,nu,d):
    tau_s = stokes_relaxation_time(d,nu,B)
    return 3 / (np.sqrt(np.pi * (1+2*B)* tau_s))
def prefactor_force(B,nu,d):
    return 9 * np.sqrt(nu) / (np.sqrt(np.pi) * B * d)

def time_oseen(uslip, nu):
    return nu / (uslip**2)

In [ ]:
# plot history kernels

fig, ax = plt.subplots()
Rep = 450
B = 0.68
factor = prefactor_force(B, kinematic_viscosity_water,diameter)
T=48*3600 # 5 hours
slist = np.arange(0,T,0.1)
ax.plot((T-slist)/3600,Basset_kernel(slist,T),color='purple')
ax.plot((T-slist)/3600,Mei_kernel(slist,T,Rep,kinematic_viscosity_water,diameter,c1_kim, c2_kim),'--',color='orange')

ax.set_yscale('log')
ax.set_xscale('log')
ax.legend(['Basset','Mei/Adrian Re$_p^{\\mathrm{In}}$='+f'{Rep}'])
ax.set_xlabel('t-s [h]')
ax.set_ylabel('H(t-s)')

fig.tight_layout()
# fig.savefig('../figures/history_force/H_mei_Basset_shape.pdf') # not save for now before you checked it


In [ ]:
# integrate kernels analytical expersion duslip/dt = 1, f = 0
    
def Basset_integral(t):
    return 2* np.sqrt(t)

def Mei_integral(t,Rep,nu,d):
    c2 = 0.2
    fh = (0.75 + c2 * Rep) 
    A = nu * Rep**2 / (fh**2 * d**2)
    return 2* np.sqrt(t) * sc.hyp2f1(5/6,5/2,11/6,-(np.pi * (A*t)**3 / 4)**(1/5))  # the hyp2f1 is the same function in mathematica and here but you make some mistake with assuming thins



## test function sine

In [ ]:
# integrate kernels now sinus signal TM2 
T_M2 = 12.4 * 3600

def Basset_sin_kernel(s,t,T):
    
    omega = 2 * np.pi / T
    return np.sin(omega * s) /np.sqrt(t-s) 

def Mei_sin_kernel(s,t,Rep,nu,d, T):
    
    omega = 2 * np.pi / T
    c1 = 2.5
    c2 = 0.2
    fh = (0.75 + c2 * Rep)
    A = nu * Rep*Rep / (fh*fh * d * d)
    term2 = ( np.sqrt(np.pi)/2 *  (A * (t - s))**(3/2) ) ** (1.0 / c1)
    return np.sin(omega * s)/np.sqrt(t-s) * ((1 + term2) ** (-c1))

# plot integrand

T=50*3600 # 5 hours
fig,ax=plt.subplots()
slist = np.arange(0,T,0.1)
Rep = 450
ax.plot((T-slist)/3600,Basset_sin_kernel(slist,T, T_M2),color='purple')
ax.plot((T-slist)/3600,Mei_sin_kernel(slist,T,Rep,kinematic_viscosity_water,diameter,T_M2),'--',color='orange')

# ax.set_yscale('log')
# ax.set_xscale('log')
ax.set_ylim(-0.01,0.01)
# ax.set_yscale('log')
ax.set_xlabel('t-s [h]')
ax.set_ylabel('H(t-s)sin($\\omega_{M2}$ s)')
fig.tight_layout()
# fig.savefig('../figures/history_force/H_mei_Basset_sine.pdf')

In [ ]:
# now use hinsberg method
def Hinsberg(f, N, h):
    """
    Calculate history term using the expression proposed by Hinsberg 2011
    with N total number of segments
    h timestep
    f array of N elements (given by slip velocity or slip + part of Mei kernel)
    note f has length N+1 k in [0,N]
    """
    K0 = 4 / 3 * f[N] * np.sqrt(h)
    KN = (
        f[0]
        * np.sqrt(h)
        * (N - 4 / 3)
        / ((N - 1) * np.sqrt(N - 1) + (N - 3 / 2) * np.sqrt(N))
    )
    KSUM = np.sum(
        [
            np.sqrt(h)
            * f[N - k]
            * (
                (k + 4 / 3) / ((k + 1) ** (3 / 2) + (k + 3 / 2) * np.sqrt(k))
                + (k - 4 / 3) / ((k - 1) ** (3 / 2) + (k - 3 / 2) * np.sqrt(k))
            )
            for k in range(1, N, 1)
        ]
    )
    return K0 + KN + KSUM


def g_Mei_kernel(s,f_u,t,Rep,nu,d, c1, c2):
    fh = (0.75 + c2 * Rep) 
    A = nu * Rep*Rep / (fh*fh * d*d)
    term2 = (np.sqrt(np.pi) / 2 *  (A * (t - s))**(3/2))**(1.0/c1)
    return f_u * ((1 + term2)**(-c1))


def g_Talaei_kernel(s,f_u,t,Rep,nu,d):
    crep = factor_drag_white1991(Rep)
    term2 = np.sqrt(nu ) *  Rep * np.sqrt(t-s)/d
    return  f_u * crep *  1/(1 + term2)

def g_Basset_kernel(s,f_u):
    return f_u

def f_u_constant(s,c):
    return np.full(s.size,c)



def f_u_sin(s,omega_M2):
    return np.sin(omega_M2 * s)

In [ ]:
# calculate history term with Basset, Mei/adrian kernel for Sine with M2 period for Rep = 450, using numerical integration and hinsberg method
dt = 1
T = 48 * 3600 
Basset_force_Hinsberg = []
Mei_force_Hinsberg = []
T_M2 = 12.4 * 3600
omega_M2 = 2 * np.pi / T_M2
twindowlist = np.arange(360,T,3600)
Rep=450

Basset_force = []
Mei_force = []

Basset_force_trap = []
Mei_force_trap = []


for twindow in twindowlist:
    # print(twindow)
    sol_B = integrate.quad(Basset_sin_kernel,T-twindow,T,args=(T, T_M2))
    sol_M =integrate.quad(Mei_sin_kernel,T-twindow,T,args=(T,Rep,kinematic_viscosity_water,diameter,T_M2)) 

    Basset_force.append(sol_B[0])
    Mei_force.append(sol_M[0])
    slist = np.arange(T-twindow,T,dt)

    # slist=np.arange(0,twindow,dt)
    fuconstant = f_u_sin(slist,omega_M2)#f_u_constant(slist,1)

    fbasset = g_Basset_kernel(slist,fuconstant)
    fmei = g_Mei_kernel(slist,fuconstant,T,Rep,kinematic_viscosity_water,diameter,c1_kim,c2_kim)

    Basset_force_Hinsberg.append(Hinsberg(fbasset,fbasset.size-1,dt))
    Mei_force_Hinsberg.append(Hinsberg(fmei,fmei.size-1,dt))

    #trapaizoid method -> does not work
    sol_trap_B = integrate.simpson(Basset_sin_kernel(slist,T,T_M2),dx=dt)
    sol_trap_M = integrate.simpson(Mei_sin_kernel(slist,T,Rep,kinematic_viscosity_water,diameter, T_M2),dx=dt)


    Basset_force_trap.append(sol_trap_B)
    Mei_force_trap.append(sol_trap_M)


fig, ax = plt.subplots()

factor=prefactor_force(B,kinematic_viscosity_water,diameter)



ax.plot(twindowlist/3600,factor*np.array(Basset_force_Hinsberg),'o',color='purple',alpha=0.3)

ax.plot(twindowlist/3600,factor*np.array(Mei_force_Hinsberg),'s',color='orange',alpha=0.3)
ax.plot(twindowlist/3600,factor*np.array(Basset_force),'--',color='purple')
ax.plot(twindowlist/3600,factor*np.array(Mei_force),'--',color='orange')
    
ax.set_ylabel('History force / (A m$_p$) ')
ax.legend(['numerical integration Basset','Hinsberg summation Basset','numerical integration Mei/Adrian','Hinsberg summation Mei/Adrian'],fontsize=18)

ax.set_xlabel('t$_{\\mathrm{window}}$ [h]')
fig.tight_layout()
# fig.savefig(f'../figures/history_force/force_sine_numerical_and_hinsberg_Rep{Rep:04d}.pdf')


## test for single particle trajectory

In [ ]:
# import data set names and directories
# settings of data
starttime = datetime(2023,9,1,0,0,0)
runtime = timedelta(days =2)
endtime = starttime + runtime
loc = 'NWES'
land_handling = 'delete_beaching'
coriolis = True
B = 0.68
tau = 3196#tau = 2994.76
Replist = [0,10,100,450,1000]
starttimes = [datetime(2023, 9, 1, 0, 0, 0, 0)]
nparticles = 52511
chunck_time = 100
coriolis = True
gradient = True
dt = timedelta(minutes=5).seconds
Tmax = 12 * 48
n_resample = 3000 
dt_resample = dt /n_resample
print(dt_resample)

sec_in_min = 60
min_in_hour = 60
sec_in_hour = sec_in_min * min_in_hour
freqmin = 60


Replist= [0,10,100,450,1000]
base_directory = '/storage/shared/oceanparcels/output_data/data_Meike/MR_advection/NWES/'
basefile_Rep_constant = (base_directory + '{particle_type}/{loc}_'
                 'start{y_s:04d}_{m_s:02d}_{d_s:02d}_'
                 'end{y_e:04d}_{m_e:02d}_{d_e:02d}_RK4_'
                 '_Rep_{Rep:04d}_B{B:04d}_tau{tau:04d}_{land_handling}_cor_{coriolis}_gradient_{gradient}_freq{freqmin:02d}min.zarr')

basefile_Rep_drag = (base_directory + '{particle_type}/{loc}_start{y_s:04d}_{m_s:02d}_{d_s:02d}'
                 '_end{y_e:04d}_{m_e:02d}_{d_e:02d}_RK4_B{B:04d}_tau{tau:04d}_{land_handling}_cor_{coriolis}_gradient_{gradient}_freq{freqmin:02d}min.zarr')


basefiles={
           'inertial_Rep_constant':basefile_Rep_constant,
           'inertial_SM_Rep_constant':basefile_Rep_constant, 
           'inertial_SM_drag_Rep':basefile_Rep_drag,
           'inertial_drag_Rep':basefile_Rep_drag}

particle_types = ['inertial_drag_Rep','inertial_Rep_constant','inertial_SM_Rep_constant']#,'inertial_Rep_constant'] # 
names = {'inertial_Rep_constant':'MR',
         'inertial_SM_Rep_constant':'SM MR', 
         'inertial_SM_drag_Rep':'SM MR flexible Re$_p$',
         'inertial_drag_Rep':'MR flexible Re$_p$'}
Rep_dict = {'inertial_Rep_constant':Replist,
         'inertial_SM_Rep_constant':Replist,
         'inertial_SM_drag_Rep':[None],
         'inertial_drag_Rep':[None]}


# import data
# create data directory
data = {}
for pt in particle_types:
    data[pt]={}


for pt in particle_types:
    for Rep in Rep_dict[pt]:
        file = basefiles[pt].format(loc=loc,
                                   y_s=starttime.year,
                                   m_s=starttime.month,
                                   d_s=starttime.day,
                                   y_e=endtime.year,
                                   m_e=endtime.month,
                                   d_e=endtime.day,
                                   B = int(B * 1000), 
                                   tau = int(tau ),
                                   land_handling = land_handling, 
                                   coriolis = coriolis,
                                   gradient = gradient,
                                   particle_type = pt,
                                   Rep = Rep,
                                   freqmin =freqmin)
        ds = xr.open_dataset(file,
                             engine='zarr',
                             chunks={'trajectory':nparticles, 'obs':chunck_time},
                             drop_variables=['B','tau','z'],
                             decode_times=False) 
        print(file)

        
        
        data[pt][Rep]= ds 

In [ ]:
def resample_time(ds, n_resample):
    # Convert obs to numpy array
    obs = ds['obs']
    obs_vals = obs.values
    # Create new obs_resampled n_resample x the original points
    obs_resampled = np.linspace(obs_vals.min(), obs_vals.max(), num=(len(obs_vals) - 1) * n_resample + 1)
    time_resampled = ds['time'].interp(obs=('obs_resampled', obs_resampled))
    ds = ds.assign_coords(obs_resampled=obs_resampled)
    ds['time_resampled'] = time_resampled
    return ds 


def cs_resample_and_derivative(v, t, tresample):
    """
    function that takes an xarray dataarray with time coordinates and returns it resampled function and derivative 
    """
    mask = ~np.isnan(v)
    cs = CubicSpline(t[mask],v[mask])
    data_resampled = cs(tresample)
    data_derivative_resampled = cs.derivative()(tresample)
    return data_resampled, data_derivative_resampled


In [ ]:
# integrand BAsset
Basset_force_dict={}

for Rep in Replist:
    Basset_force_dict[Rep]={}
    # fit cubic spline through hourly data
    nmax = 47+1#-1
    pt = 'inertial_Rep_constant'
    id =25
    omega_earth =  7.2921e-5 #[rad/sec]

    ds_sel= data[pt][Rep].isel(trajectory=id,obs=slice(1,nmax))
    f_rotation = 2 * omega_earth * np.sin(np.pi * ds_sel.lat[-1].values /180)
    uslip = ds_sel.uslip
    vslip = ds_sel.vslip
    t = (ds_sel.time).values
    n_resample = 3600 # now resampled every second
    tresample = np.linspace(t.min(), t.max(), num=(len(t) - 1) * n_resample + 1)
    tmax=tresample[-1]
    cs_u = CubicSpline(t,uslip)
    cs_v = CubicSpline(t,vslip)
    u_resampled = cs_u(tresample)
    u_derivative_resampled = cs_u.derivative()(tresample)
    v_resampled = cs_v(tresample)
    v_derivative_resampled = cs_v.derivative()(tresample)
    
    Basset_force_x_Rep_flexible_list= []
    Basset_force_y_Rep_flexible_list = []
    dt = 1
    twindowlist = np.arange(3600,tmax,3600)
    for twindow in twindowlist:
        slist = np.arange(tmax-twindow,tmax+0.5*dt,dt)
        nstart = int((slist[0]-t[0])/dt)
        nend = nstart + int(twindow/dt)+1# int((slist[-1]+0.5*dt)/dt)+1
        selection_x = (u_derivative_resampled-v_resampled*f_rotation)[nstart:nend]
        selection_y = (v_derivative_resampled+u_resampled*f_rotation)[nstart:nend]
   
        uslip_resampled = np.sqrt(u_resampled**2 + v_resampled**2)[nstart:nend] 
        repvalues = Re_particle(uslip_resampled,diameter, kinematic_viscosity_water)
        fbasset_x = g_Basset_kernel(slist,selection_x)#,Rep,kinematic_viscosity_water,diameter)
        fbasset_y = g_Basset_kernel(slist,selection_y)#,Rep,kinematic_viscosity_water,diameter)
        Basset_force_x_Rep_flexible_list.append(Hinsberg(fbasset_x,fbasset_x.size-1,dt))
        Basset_force_y_Rep_flexible_list.append(Hinsberg(fbasset_y,fbasset_y.size-1,dt))


 
    Basset_force_dict[Rep]=np.sqrt(np.array(Basset_force_x_Rep_flexible_list)**2+np.array(Basset_force_y_Rep_flexible_list)**2)
    print(Rep)


# flexible Rep 
Rep = None
Basset_force_dict[Rep]={}
# fit cubic spline through hourly data
nmax = 47+1#-1
pt = 'inertial_drag_Rep'
id =25
omega_earth =  7.2921e-5 #[rad/sec]

ds_sel= data[pt][Rep].isel(trajectory=id,obs=slice(1,nmax))
f_rotation = 2 * omega_earth * np.sin(np.pi * ds_sel.lat[-1].values /180)
uslip = ds_sel.uslip
vslip = ds_sel.vslip
t = (ds_sel.time).values
n_resample = 3600 # now resampled every second
tresample = np.linspace(t.min(), t.max(), num=(len(t) - 1) * n_resample + 1)
tmax = tresample[-1]
cs_u = CubicSpline(t,uslip)
cs_v = CubicSpline(t,vslip)
u_resampled = cs_u(tresample)
u_derivative_resampled = cs_u.derivative()(tresample)
v_resampled = cs_v(tresample)
v_derivative_resampled = cs_v.derivative()(tresample)

Basset_force_x_Rep_flexible_list= []
Basset_force_y_Rep_flexible_list = []
dt = 1    
twindowlist = np.arange(3600,tmax,3600)
for twindow in twindowlist:
    slist = np.arange(tmax-twindow,tmax+0.5*dt,dt)
    nstart = int((slist[0]-t[0])/dt)
    nend = nstart + int(twindow/dt)+1# int((slist[-1]+0.5*dt)/dt)+1
    selection_x = (u_derivative_resampled-v_resampled*f_rotation)[nstart:nend]
    selection_y = (v_derivative_resampled+u_resampled*f_rotation)[nstart:nend]
    uslip_resampled = np.sqrt(u_resampled**2 + v_resampled**2)[nstart:nend] 
    repvalues = Re_particle(uslip_resampled,diameter, kinematic_viscosity_water)
    fbasset_x = g_Basset_kernel(slist,selection_x)#,Rep,kinematic_viscosity_water,diameter)
    fbasset_y = g_Basset_kernel(slist,selection_y)#,Rep,kinematic_viscosity_water,diameter)
    Basset_force_x_Rep_flexible_list.append(Hinsberg(fbasset_x,fbasset_x.size-1,dt))
    Basset_force_y_Rep_flexible_list.append(Hinsberg(fbasset_y,fbasset_y.size-1,dt))



Basset_force_dict[Rep]=np.sqrt(np.array(Basset_force_x_Rep_flexible_list)**2+np.array(Basset_force_y_Rep_flexible_list)**2)
print(Rep)

In [ ]:
# plot Basset force af function of window
fig, ax = plt.subplots()
legend = []
for Rep,color in zip(Replist,Magmalist):
    factor = 3 / np.sqrt(np.pi * (1+2 * B)* tau) #prefactor(B=B,nu=kinematic_viscosity_water,d=diameter)
    ax.plot(twindowlist[:-1]/3600,Basset_force_dict[Rep][:-1]/(Basset_force_dict[Rep][-2]),'-.',color=color)
    legend.append(f'c({Rep})')
factor = 3 / np.sqrt(np.pi * (1+2 * B)* tau)
ax.plot(twindowlist[:-1]/3600,Basset_force_dict[None][:-1]/(Basset_force_dict[None][-2]),'--',color='cornflowerblue')
legend.append('c$_p$(t)')
ax.set_ylabel('F$_{\\mathrm{Basset}}$ (t$_{\\mathrm{window}}$)/F$_{\\mathrm{Basset}}$ (48 h)]')
ax.set_xlabel('t$_{\\mathrm{window}}$ [h]')
ax.legend(legend,fontsize=20)
# ax.set_yscale('log')

In [ ]:
# calculate Mei history force
Mei_force_dict={}

for Rep in Replist:
    Mei_force_dict[Rep]={}
    # fit cubic spline through hourly data
    nmax = 47+1#-1
    pt = 'inertial_Rep_constant'
    id =25
    omega_earth =  7.2921e-5 #[rad/sec]

    ds_sel= data[pt][Rep].isel(trajectory=id,obs=slice(1,nmax))
    f_rotation = 2 * omega_earth * np.sin(np.pi * ds_sel.lat[-1].values /180)
    uslip = ds_sel.uslip
    vslip = ds_sel.vslip
    t = (ds_sel.time).values
    n_resample = 3600 # now resampled every second
    tresample = np.linspace(t.min(), t.max(), num=(len(t) - 1) * n_resample + 1)
    tmax=tresample[-1]
    cs_u = CubicSpline(t,uslip)
    cs_v = CubicSpline(t,vslip)
    u_resampled = cs_u(tresample)
    u_derivative_resampled = cs_u.derivative()(tresample)
    v_resampled = cs_v(tresample)
    v_derivative_resampled = cs_v.derivative()(tresample)
    
    Mei_force_x_Rep_flexible_list= []
    Mei_force_y_Rep_flexible_list = []
    dt = 1
    twindowlist = np.arange(3600,tmax,3600)
    for twindow in twindowlist:
        slist = np.arange(tmax-twindow,tmax+0.5*dt,dt)
        nstart = int((slist[0]-t[0])/dt)
        nend = nstart + int(twindow/dt)+1# int((slist[-1]+0.5*dt)/dt)+1
        # print(nend-nstart)
        # print(slist.size)
        selection_x = (u_derivative_resampled-v_resampled*f_rotation)[nstart:nend]
        selection_y = (v_derivative_resampled+u_resampled*f_rotation)[nstart:nend]
        # print(selection_x.size)
        uslip_resampled = np.sqrt(u_resampled**2 + v_resampled**2)[nstart:nend] 
        repvalues = Rep # Re_particle(uslip_resampled,diameter, kinematic_viscosity_water)
        fmei_x_f = g_Mei_kernel(slist,selection_x,tmax,repvalues,kinematic_viscosity_water,diameter,c1_kim,c2_kim)
        fmei_y_f = g_Mei_kernel(slist,selection_y,tmax,repvalues,kinematic_viscosity_water,diameter,c1_kim,c2_kim)
        Mei_force_x_Rep_flexible_list.append(Hinsberg(fmei_x_f,fmei_x_f.size-1,dt))
        Mei_force_y_Rep_flexible_list.append(Hinsberg(fmei_y_f,fmei_y_f.size-1,dt))


 
    Mei_force_dict[Rep]=np.sqrt(np.array(Mei_force_x_Rep_flexible_list)**2+np.array(Mei_force_y_Rep_flexible_list)**2)
    print(Rep)


# flexible Rep 
Rep = None
Mei_force_dict[Rep]={}
# fit cubic spline through hourly data
nmax = 47+1#-1
pt = 'inertial_drag_Rep'
id =25
omega_earth =  7.2921e-5 #[rad/sec]

ds_sel= data[pt][Rep].isel(trajectory=id,obs=slice(1,nmax))
f_rotation = 2 * omega_earth * np.sin(np.pi * ds_sel.lat[-1].values /180)
uslip = ds_sel.uslip
vslip = ds_sel.vslip
t = (ds_sel.time).values
n_resample = 3600 # now resampled every second
tresample = np.linspace(t.min(), t.max(), num=(len(t) - 1) * n_resample + 1)
tmax=tresample[-1]
cs_u = CubicSpline(t,uslip)
cs_v = CubicSpline(t,vslip)
u_resampled = cs_u(tresample)
u_derivative_resampled = cs_u.derivative()(tresample)
v_resampled = cs_v(tresample)
v_derivative_resampled = cs_v.derivative()(tresample)

Mei_force_x_Rep_flexible_list= []
Mei_force_y_Rep_flexible_list = []
dt = 1
twindowlist = np.arange(3600,tmax,3600)
for twindow in twindowlist:
    slist = np.arange(tmax-twindow,tmax+0.5*dt,dt)
    nstart = int((slist[0]-t[0])/dt)
    nend = nstart + int(twindow/dt)+1# int((slist[-1]+0.5*dt)/dt)+1
    selection_x = (u_derivative_resampled-v_resampled*f_rotation)[nstart:nend]
    selection_y = (v_derivative_resampled+u_resampled*f_rotation)[nstart:nend]

    uslip_resampled = np.sqrt(u_resampled**2 + v_resampled**2)[nstart:nend] 
    repvalues = Re_particle(uslip_resampled,diameter, kinematic_viscosity_water)
    fmei_x_f = g_Mei_kernel(slist,selection_x,tmax,repvalues,kinematic_viscosity_water,diameter,c1_kim,c2_kim)
    fmei_y_f = g_Mei_kernel(slist,selection_y,tmax,repvalues,kinematic_viscosity_water,diameter,c1_kim,c2_kim)
    Mei_force_x_Rep_flexible_list.append(Hinsberg(fmei_x_f,fmei_x_f.size-1,dt))
    Mei_force_y_Rep_flexible_list.append(Hinsberg(fmei_y_f,fmei_y_f.size-1,dt))



Mei_force_dict[Rep]=np.sqrt(np.array(Mei_force_x_Rep_flexible_list)**2+np.array(Mei_force_y_Rep_flexible_list)**2)
print(Rep)

In [ ]:
twindowlist = np.arange(3600,tmax,3600)#
print(tmax/3600)
legend = []
fig = plt.figure()
h = [Size.Fixed(1.0), Size.Fixed(8)]
v = [Size.Fixed(0.7), Size.Fixed(6)]
divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)
ax = fig.add_axes(divider.get_position(),
                    axes_locator=divider.new_locator(nx=1, ny=1))
for Rep,color in zip(Replist,Magmalist):
    ax.plot(twindowlist/3600,Mei_force_dict[Rep]/Mei_force_dict[Rep][19],'-.',color=color)
    legend.append(f'c({Rep})')
ax.plot(twindowlist/3600,Mei_force_dict[None]/Mei_force_dict[None][19],'--',color='cornflowerblue')
legend.append('c$_p$(t)')
ax.set_ylabel('F$_{\\mathrm{Mei/Adrian}}$ (t$_{\\mathrm{window}}$)/F$_{\\mathrm{Mei/Adrian}}$ (20 h) ')
ax.set_xlabel('t$_{\\mathrm{window}}$ [h]')
ax.legend(legend,fontsize=18)

# fig.savefig('../figures/history_force/fmei_adrian_twindow_id25.pdf')



In [ ]:


twindowlist = np.arange(3600,tmax,3600)#
print(tmax/3600)
legend = []
fig = plt.figure()
h = [Size.Fixed(1.0), Size.Fixed(8)]
v = [Size.Fixed(0.7), Size.Fixed(6)]
divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)
ax = fig.add_axes(divider.get_position(),
                    axes_locator=divider.new_locator(nx=1, ny=1))
for Rep,color in zip(Replist,Magmalist):
    ax.plot(twindowlist/3600,Mei_force_dict[Rep]/Mei_force_dict[Rep][19],'-.',color=color)
    legend.append(f'c({Rep})')
ax.plot(twindowlist/3600,Mei_force_dict[None]/Mei_force_dict[None][19],'--',color='cornflowerblue')
legend.append('c$_p$(t)')
ax.set_ylabel('F$_{\\mathrm{Mei/Adrian}}$ (t$_{\\mathrm{window}}$)/F$_{\\mathrm{Mei/Adrian}}$ (20 h) ')
ax.set_xlabel('t$_{\\mathrm{window}}$ [h]')
ax.legend(legend,fontsize=18)

fig.savefig('../figures/history_force/fmei_adrian_twindow_id25.pdf')

fig, ax = plt.subplots()
legend = []
for Rep,color in zip(Replist,Magmalist):
    ax.plot(twindowlist[:-3]/3600,1-Mei_force_dict[Rep][:-3]/Mei_force_dict[Rep][-2],'--',color=color)
    legend.append(f'c({Rep})')
ax.plot(twindowlist[:-3]/3600,1-Mei_force_dict[None][:-3]/Mei_force_dict[None][-2],'-',color='cornflowerblue')
legend.append('c$_p$(t)')
ax.set_ylabel('1-$F_{\\mathrm{Mei/Adrian}} (t_{\\mathrm{window}})/F_{\\mathrm{Mei/Adrian}}(45\\mathrm{h})$')
ax.set_xlabel('t$_{\\mathrm{window}}$ [h]')
ax.legend(legend,fontsize=20)
# ax.set_yscale('log')



In [ ]:
# check effect of dt used 
# integrand 
Mei_force_dict_dt = {}



for Rep in Replist:
    Mei_force_dict_dt[Rep]={}
    nmax = 47+1#-1
    pt = 'inertial_Rep_constant'
    id =25
    omega_earth =  7.2921e-5 #[rad/sec]

    ds_sel= data[pt][Rep].isel(trajectory=id,obs=slice(1,nmax))
    f_rotation = 2 * omega_earth * np.sin(np.pi * ds_sel.lat[-1].values /180)
    uslip = ds_sel.uslip
    vslip = ds_sel.vslip
    t = (ds_sel.time).values
    n_resample = 3600 # now resampled every second
    tresample = np.linspace(t.min(), t.max(), num=(len(t) - 1) * n_resample + 1)
    tmax=tresample[-1]
    cs_u = CubicSpline(t,uslip)
    cs_v = CubicSpline(t,vslip)


    # data_derivative_resampled.size
    Mei_force_x_list_dt = []
    Mei_force_y_list_dt = []


    twindowlist = np.arange(360,tmax+1,360)#(logspace(-1,5,3000)
    dtlist =np.array([1/10,1/5,1/2,1,10,60,300,600])

    for dt in dtlist:
        nresample = int(3600/dt)
        tresample = np.linspace(t.min(), t.max(), num=(len(t) - 1) * nresample + 1)
        u_slip_resampled = cs_u(tresample)
        v_slip_resampled = cs_v(tresample)

        u_slip_derivative_resampled =cs_u.derivative()(tresample)
        v_slip_derivative_resampled =cs_u.derivative()(tresample)


        slist = tresample-t[0]#np.arange(0,tmax+0.5*dt,dt)

        # print(nend-nstart)
        # print(slist.size)
        selection_x = (u_slip_derivative_resampled-v_slip_resampled*f_rotation)
        selection_y = (v_slip_derivative_resampled-u_slip_resampled*f_rotation)
        uslip_resampled = np.sqrt(u_slip_resampled**2 + v_slip_resampled**2)
        repvalues = Rep# Re_particle(uslip_resampled,diameter, kinematic_viscosity_water)
        fmei_x = g_Mei_kernel(slist,selection_x,tmax,repvalues,kinematic_viscosity_water,diameter, c1_kim, c2_kim)
        fmei_y = g_Mei_kernel(slist,selection_y,tmax,repvalues,kinematic_viscosity_water,diameter,c1_kim, c2_kim)


        Mei_force_x_list_dt.append(Hinsberg(fmei_x,fmei_x.size-1,dt))
        Mei_force_y_list_dt.append(Hinsberg(fmei_y,fmei_y.size-1,dt))


    Mei_force_dict_dt[Rep]=np.sqrt(np.array(Mei_force_x_list_dt)**2+np.array(Mei_force_y_list_dt)**2)


# flexible Rep 
Rep = None
nmax = 47+1#-1
pt = 'inertial_drag_Rep'
id =25
omega_earth =  7.2921e-5 #[rad/sec]

ds_sel= data[pt][Rep].isel(trajectory=id,obs=slice(1,nmax))
f_rotation = 2 * omega_earth * np.sin(np.pi * ds_sel.lat[-1].values /180)
uslip = ds_sel.uslip
vslip = ds_sel.vslip
t = (ds_sel.time).values
n_resample = 3600 # now resampled every second
tresample = np.linspace(t.min(), t.max(), num=(len(t) - 1) * n_resample + 1)
tmax=tresample[-1]
cs_u = CubicSpline(t,uslip)
cs_v = CubicSpline(t,vslip)


# data_derivative_resampled.size
Mei_force_x_list_dt = []
Mei_force_y_list_dt = []


twindowlist = np.arange(360,tmax+1,360)#(logspace(-1,5,3000)
dtlist =np.array([1/10,1/5,1/2,1,10,60,300,600])

for dt in dtlist:
    nresample = int(3600/dt)
    tresample = np.linspace(t.min(), t.max(), num=(len(t) - 1) * nresample + 1)
    u_slip_resampled = cs_u(tresample)
    v_slip_resampled = cs_v(tresample)
    u_slip_derivative_resampled =cs_u.derivative()(tresample)
    v_slip_derivative_resampled =cs_u.derivative()(tresample)

    slist = tresample-t[0]
    # slist = np.arange(0,tmax+0.5*dt,dt)

    # print(nend-nstart)
    # print(slist.size)
    selection_x = (u_slip_derivative_resampled-v_slip_resampled*f_rotation)
    selection_y = (v_slip_derivative_resampled-u_slip_resampled*f_rotation)
    uslip_resampled = np.sqrt(u_slip_resampled**2 + v_slip_resampled**2)
    repvalues = Re_particle(uslip_resampled,diameter, kinematic_viscosity_water)
    fmei_x = g_Mei_kernel(slist,selection_x,tmax,repvalues,kinematic_viscosity_water,diameter, c1_kim, c2_kim)
    fmei_y = g_Mei_kernel(slist,selection_y,tmax,repvalues,kinematic_viscosity_water,diameter,c1_kim, c2_kim)


    Mei_force_x_list_dt.append(Hinsberg(fmei_x,fmei_x.size-1,dt))
    Mei_force_y_list_dt.append(Hinsberg(fmei_y,fmei_y.size-1,dt))


Mei_force_dict_dt[Rep]=np.sqrt(np.array(Mei_force_x_list_dt)**2+np.array(Mei_force_y_list_dt)**2)

In [ ]:
fig = plt.figure()
h = [Size.Fixed(1.0), Size.Fixed(8)]
v = [Size.Fixed(0.7), Size.Fixed(6)]
divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)
ax = fig.add_axes(divider.get_position(),
                    axes_locator=divider.new_locator(nx=1, ny=1))
legend = []
for Rep, color in zip(Replist,Magmalist):
    ax.plot(dtlist,Mei_force_dict_dt[Rep]/Mei_force_dict_dt[Rep][0],'-.o',color=color)
    legend.append(f'c({Rep})')
ax.plot(dtlist,Mei_force_dict_dt[None]/Mei_force_dict_dt[None][0],'--s',color='cornflowerblue')
legend.append('c$_p$(t)')
ax.legend(legend,fontsize=18)
ax.set_ylabel('F$_{\\mathrm{Mei/Adrian}}$ ($\\Delta$t)/F$_{\\mathrm{Mei/Adrian}}$ (0.1 s) ')
ax.set_xlabel('$\\Delta$t [s]')
ax.set_xscale('log')
# ax.set_yscale('log')
# ax.set_title('Mei/Adrian history force',fontsize=18)

# fig.tight_layout()
fig.savefig(f'../figures/history_force/mei_adrian_history_force_dt.pdf')

In [ ]:
# check effect of dt used for basset short window
# integrand 
Basset_force_dict_dt = {}


tmax=tresample[-1]
for Rep in Replist:
    Basset_force_dict_dt[Rep]={}
    nmax = 47+1#-1
    pt = 'inertial_Rep_constant'
    id =25
    omega_earth =  7.2921e-5 #[rad/sec]

    ds_sel= data[pt][Rep].isel(trajectory=id,obs=slice(1,nmax))
    f_rotation = 2 * omega_earth * np.sin(np.pi * ds_sel.lat[-1].values /180)
    uslip = ds_sel.uslip
    vslip = ds_sel.vslip
    t = (ds_sel.time).values
    n_resample = 3600 # now resampled every second
    tresample = np.linspace(t.min(), t.max(), num=(len(t) - 1) * n_resample + 1)
    cs_u = CubicSpline(t,uslip)
    cs_v = CubicSpline(t,vslip)


    # data_derivative_resampled.size
    Basset_force_x_list_dt = []
    Basset_force_y_list_dt = []


    # twindowlist = np.arange(360,tmax+1,360)#(logspace(-1,5,3000)
    dtlist =np.array([1/10,1/5,1/2,1,10,60,300,600])
    twindow=180
    for dt in dtlist:
        nresample = int(3600/dt)
        tresample = np.linspace(t.min(), t.max(), num=(len(t) - 1) * nresample + 1)
        u_slip_resampled = cs_u(tresample)
        v_slip_resampled = cs_v(tresample)
        u_slip_derivative_resampled =cs_u.derivative()(tresample)
        v_slip_derivative_resampled =cs_u.derivative()(tresample)


        slist = tresample-t[0]

        # print(nend-nstart)
        # print(slist.size)
        selection_x = (u_slip_derivative_resampled-v_slip_resampled*f_rotation)
        selection_y = (v_slip_derivative_resampled-u_slip_resampled*f_rotation)
        uslip_resampled = np.sqrt(u_slip_resampled**2 + v_slip_resampled**2)
        repvalues = Re_particle(uslip_resampled,diameter, kinematic_viscosity_water)
        fbasset_x = g_Basset_kernel(slist,selection_x)#(slist,selection_x,tmax,repvalues,kinematic_viscosity_water,diameter, c1_kim, c2_kim)
        fbasset_y = g_Basset_kernel(slist,selection_y)#g_Mei_kernel(slist,selection_y,tmax,repvalues,kinematic_viscosity_water,diameter,c1_kim, c2_kim)


        Basset_force_x_list_dt.append(Hinsberg(fbasset_x,fbasset_x.size-1,dt))
        Basset_force_y_list_dt.append(Hinsberg(fbasset_y,fbasset_y.size-1,dt))


    Basset_force_dict_dt[Rep]=np.sqrt(np.array(Basset_force_x_list_dt)**2+np.array(Basset_force_y_list_dt)**2)


# flexible Rep 
Rep = None
nmax = 47+1#-1
pt = 'inertial_drag_Rep'
id =25
omega_earth =  7.2921e-5 #[rad/sec]

ds_sel= data[pt][Rep].isel(trajectory=id,obs=slice(1,nmax))
f_rotation = 2 * omega_earth * np.sin(np.pi * ds_sel.lat[-1].values /180)
uslip = ds_sel.uslip
vslip = ds_sel.vslip
t = (ds_sel.time).values
n_resample = 3600 # now resampled every second
tresample = np.linspace(t.min(), t.max(), num=(len(t) - 1) * n_resample + 1)
cs_u = CubicSpline(t,uslip)
cs_v = CubicSpline(t,vslip)


# data_derivative_resampled.size
Basset_force_x_list_dt = []
Basset_force_y_list_dt = []


# twindowlist = np.arange(360,tmax+1,360)#(logspace(-1,5,3000)
twindow=180
dtlist =np.array([1/10,1/5,1/2,1,10,60,300,600])

for dt in dtlist:
    nresample = int(3600/dt)
    tresample = np.linspace(t.min(), t.max(), num=(len(t) - 1) * nresample + 1)
    u_slip_resampled = cs_u(tresample)
    v_slip_resampled = cs_v(tresample)
    u_slip_derivative_resampled =cs_u.derivative()(tresample)
    v_slip_derivative_resampled =cs_u.derivative()(tresample)


    slist = tresample-t[0]

    # print(nend-nstart)
    # print(slist.size)
    selection_x = (u_slip_derivative_resampled-v_slip_resampled*f_rotation)
    selection_y = (v_slip_derivative_resampled-u_slip_resampled*f_rotation)
    uslip_resampled = np.sqrt(u_slip_resampled**2 + v_slip_resampled**2)
    repvalues = Re_particle(uslip_resampled,diameter, kinematic_viscosity_water)
    fbasset_x = g_Basset_kernel(slist,selection_x)#(slist,selection_x,tmax,repvalues,kinematic_viscosity_water,diameter, c1_kim, c2_kim)
    fbasset_y = g_Basset_kernel(slist,selection_y)#g_Mei_kernel(slist,selection_y,tmax,repvalues,kinematic_viscosity_water,diameter,c1_kim, c2_kim)


    Basset_force_x_list_dt.append(Hinsberg(fbasset_x,fbasset_x.size-1,dt))
    Basset_force_y_list_dt.append(Hinsberg(fbasset_y,fbasset_y.size-1,dt))


Basset_force_dict_dt[Rep]=np.sqrt(np.array(Basset_force_x_list_dt)**2+np.array(Basset_force_y_list_dt)**2)

In [ ]:
fig, ax = plt.subplots()
legend = []
for Rep, color in zip(Replist,Magmalist):
    print(Basset_force_dict_dt[Rep][0])
    ax.plot(dtlist[0:4],Basset_force_dict_dt[Rep][0:4]/Basset_force_dict_dt[Rep][0],'--s',color=color)
    legend.append('Re$_p^{\\mathrm{In}}$='+f'{Rep}')
ax.plot(dtlist[0:4],Basset_force_dict_dt[None][0:4]/Basset_force_dict_dt[None][0],'--s',color='cornflowerblue')
legend.append('Flexible Re$_p$')
ax.legend(legend)
ax.set_ylabel('history force / history force ($\\Delta$t = 0.1 s)')
ax.set_xlabel('$\\Delta$t [s]')
ax.ticklabel_format(axis='y',style='plain')#,scilimits=(-4,4))
# ax.set_xscale('log')

# fig.tight_layout()
# fig.savefig(f'../figures/history_force/history_force_dt-rep{Rep:04d}.pdf')